# contrastive_finetune: contrastive fine-tuning on top of the distilled main-line student

Start: the 1k-base student distilled from BGE-M3 + instructed 8K-embed (10,800 steps, lr 1e-4; TR-MTEB 67.71). Training: in-batch-negative InfoNCE on the (question, passage) pairs of the three QA datasets (`configs/contrastive_after_distill.yaml`), one epoch (~300 steps, a few minutes on an A100), one run per learning rate in `LRS`. Afterwards each result is blended with the distilled model (weight average, `turkish_distill.soup`) at the contrastive shares in `SHARES`, since the main risk is forgetting what distillation gave. The student sees plain text (no instruction) everywhere.

Nothing here is a result: untuned starting values. Evaluate with `eval_trio` (TR-MTEB), Mizan and the RAG notebooks; compare against 67.71 and read only differences above ~0.3 (seed noise ~0.1, single seed). The step-0 `val/mrr@10` in `metrics.jsonl` is the distilled model's own pair-retrieval score (a sanity check, not a selection metric: it did not track TR-MTEB before). Re-running after a disconnect resumes the unfinished run and skips finished runs and soups. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)


# Contrastive runs: one per learning rate (the distillation rate was 1e-4). A later override wins.
DISTILLED = f'{DRIVE_ROOT}/runs/scale_two_instr8k_s10800_lr1e4/20261027-152516-b2b0c4b/final'
CONFIG = 'configs/contrastive_after_distill.yaml'
LRS = {'1e5': 1e-5, '2e5': 2e-5}          # label -> learning rate
SHARES = [0.3, 0.5, 0.7]                  # share of the contrastive model in each soup (the rest is the distilled model)

import glob, os

assert os.path.isdir(DISTILLED), f'distilled model not found: {DISTILLED}'
finals = {}
for label, lr in LRS.items():
    name = f'ctr_after_distill_lr{label}'
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    done = [d for d in run_dirs if os.path.isdir(d + 'final')]
    if done:
        print();print(f'########## {name}: already finished, skipped')
        finals[label] = done[-1] + 'final'
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}', f'student.hf_id={DISTILLED}', f'train.lr={lr}'])
    print();print(f'########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train_contrastive --config {CONFIG} {run_id_arg} {overrides}')
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    finals[label] = run_dirs[-1] + 'final'
print(finals)


In [ ]:
# Soups: distilled model blended with each contrastive model; saved to Drive/models/ (evaluate with eval_trio presets).
from turkish_distill.soup import soup_models

for label, final in finals.items():
    for share in SHARES:
        out = f'{DRIVE_ROOT}/models/soup_ctr_lr{label}_c{int(round(share * 100))}'
        if os.path.exists(out + '/SOUP.json'):
            print('exists, skipped:', out)
            continue
        soup_models([DISTILLED, final], [1 - share, share], out, label=f'distilled + contrastive lr {label}')
        print('saved:', out)


In [ ]:
# Paste these into eval_trio's PRESETS (they are written out here so the paths need no guessing):
for label, final in finals.items():
    print(f"    'ctr_lr{label}': dict(model='{final}'),")
    for share in SHARES:
        print(f"    'soup_ctr_lr{label}_c{int(round(share * 100))}': dict(model=f'{{DRIVE_ROOT}}/models/soup_ctr_lr{label}_c{int(round(share * 100))}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()
